# 00 · How to use these notebooks

After this notebook you will have a working environment, know which LLM the notebooks are talking to (and how to switch it), have checked that every library imports, and have a study track to follow.

**Time:** ~15 min · **Runs:** offline on CPU in about a minute (the version check imports every library once, and that is the slow part) · **Needs:** nothing. Start here.

## Why this matters in interviews

- The code in these notebooks is the code you would ship: a real `openai.OpenAI` client that can point at OpenAI, a local Ollama model, or a free offline stand-in. You can explain provider switching from experience, not from a slide.
- "How do you test LLM code in CI without paying for it?" is a common question. The offline stand-in used here is one concrete answer.
- Keeping keys out of git, pinning versions and making runs reproducible are what separate a demo from a product. Interviewers probe these.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo21` | What is an LLM API call actually made of, and what are you charged for? |
| `tl11` | What does Ollama do, and would you use it in production? |
| `pd16` | How would you migrate from one LLM provider to another? |
| `pd25` | How do you manage secrets and credentials for an agent? |
| `pd29` | How do you test an LLM feature in CI without spending a fortune? |

## What this folder is

The repo already explains things: interactive theory courses, plus [`ai_interview_prep`](../../ai_interview_prep/index.html) with 578 interview questions answered in theory. This folder is the **practical companion**. It is runnable code that makes those answers concrete: toy data, CPU only, about a minute per notebook, free by default.

```
ai_interview_prep/            578 questions with theory answers, IDs like `rg08`
        |
        |   every notebook opens with a table of the question IDs it makes practical
        v
jupyter_notebooks/            <- you are here: run the answer, change it, break it
        |
        '-- INTERVIEW_MAP.md  the reverse map: any question ID -> the notebooks that practise it
```

[`INTERVIEW_MAP.md`](../INTERVIEW_MAP.md) sits at the root of this folder. When a theory answer feels abstract, look up its ID there and open the notebook.

Every notebook has the same shape:

| Part | What it gives you |
|---|---|
| Why this matters in interviews | the reason to care, in 2 to 4 bullets |
| Interview questions this notebook answers | the theory IDs it makes concrete |
| Concept sections | plain English first, then the mechanism, then code, and an `assert` that checks each claim |
| Try it yourself | small exercises, each with a solution cell below it (try first) |
| Say it in an interview | the 30-second answer, the numbers, the follow-ups and traps |
| Next | where to go from here |

## Setup on Windows (once)

Open a terminal (PowerShell or cmd) in the `jupyter_notebooks` folder:

```bat
python -m venv .venv
.venv\Scripts\activate
pip install torch --index-url https://download.pytorch.org/whl/cpu
pip install -r requirements.txt
python -m ipykernel install --user --name study-notebooks --display-name "Study Notebooks (.venv)"
```

- **Install the CPU wheel of PyTorch first.** The default `pip install torch` on Windows can pull a CUDA build of roughly 2 GB. The CPU wheel from `download.pytorch.org/whl/cpu` is a fraction of that, and every notebook here runs on CPU.
- `requirements.txt` (at the root of this folder) pins every library the notebooks import, with upper bounds so a fresh install does not jump a major version.
- If PowerShell says *running scripts is disabled on this system*, run `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned` once, or use cmd.

**The kernel.** The last setup line registers the venv as a Jupyter kernel called **Study Notebooks (.venv)**, and every notebook in this folder asks for that kernel by name, so VS Code and JupyterLab select it automatically. If a notebook ever fails with `ModuleNotFoundError: No module named 'matplotlib'` (or numpy, torch, ...), it is running on your global Python instead: click the kernel name at the top right (VS Code: **Select Kernel > Jupyter Kernel**) and choose **Study Notebooks (.venv)**. The first code cell below prints `sys.executable`, which should end in `jupyter_notebooks\.venv\Scripts\python.exe`.

| Editor | How |
|---|---|
| VS Code | Install the *Python* and *Jupyter* extensions and open a notebook; it picks **Study Notebooks (.venv)**. The repo's `.vscode/settings.json` also makes the venv the default interpreter. |
| JupyterLab | With the venv activated: `pip install jupyterlab`, then `jupyter lab`, and choose **Study Notebooks (.venv)** under **Kernel > Change Kernel** if it isn't already selected. |

**Run a notebook.** Use **Run All**. If you edited cells or something looks odd, use **Restart** and then **Run All**. A notebook is only trustworthy when it runs top to bottom in a fresh kernel. The next cell checks which Python this kernel is using.

In [ ]:
import platform
import sys
from pathlib import Path

in_venv = sys.prefix != sys.base_prefix
print(f"python       : {sys.version.split()[0]} on {platform.system()}")
print(f"executable   : {sys.executable}")
print(f"inside a venv: {in_venv}")
print(f"notebook cwd : {Path.cwd().name}/")

assert sys.version_info >= (3, 11), "these notebooks need Python 3.11 or newer"
if not in_venv:
    print("WARNING: this kernel is not a virtual environment - select the .venv kernel (see the table above).")

### The predict-then-run habit

Outputs are **not saved** in these files. That is deliberate. Before you run a cell, say what it will print. Then run it. When the output surprises you, you have found the thing to learn, and you will remember it far better than something you only read.

Try it now. Predict all three lines, then run:

In [ ]:
print(0.1 + 0.2 == 0.3)        # floats are binary fractions: 0.1 has no exact representation
print(round(2.5), round(3.5))  # Python rounds halves to the nearest EVEN number
grid = [[0] * 2] * 2           # two rows ... or one row listed twice?
grid[0][0] = 1
print(grid)

assert (0.1 + 0.2 == 0.3) is False
assert (round(2.5), round(3.5)) == (2, 4)
assert grid == [[1, 0], [1, 0]]   # both "rows" are the same list object

## The environment check

The next cell **imports** every key library and prints a version table. It imports each one rather than just reading version metadata, because an import catches broken installs too, such as a torch DLL that fails to load on Windows. Anything missing is marked in the table and the cell keeps going.

Import times are *incremental*. `transformers` and `peft` reuse the `torch` that is already loaded, so whichever heavy library comes first pays most of the cost. A second import in the same kernel is instant.

In [ ]:
import importlib
import importlib.metadata as md
import os
import time

os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")   # litellm: use its bundled price table, no network call at import

# (what to import, pip distribution name, what the notebooks use it for)
PACKAGES = [
    ("numpy", "numpy", "arrays, vector maths"),               ("pandas", "pandas", "tables, logs"),
    ("matplotlib", "matplotlib", "charts"),                  ("sklearn", "scikit-learn", "classical ML"),
    ("scipy", "scipy", "stats"),                             ("torch", "torch", "tensors, training"),
    ("transformers", "transformers", "Hugging Face models"),  ("peft", "peft", "LoRA / QLoRA"),
    ("datasets", "datasets", "HF datasets"),                 ("bitsandbytes", "bitsandbytes", "8-/4-bit quantization"),
    ("sentence_transformers", "sentence-transformers", "embeddings, rerankers"),
    ("openai", "openai", "the LLM client"),                  ("tiktoken", "tiktoken", "token counting"),
    ("langchain", "langchain", "LangChain"),                 ("langchain_core", "langchain-core", "LangChain core"),
    ("langchain_openai", "langchain-openai", "ChatOpenAI"),  ("langgraph.graph", "langgraph", "agent graphs"),
    ("rank_bm25", "rank-bm25", "BM25 keyword search"),       ("faiss", "faiss-cpu", "vector search"),
    ("fastapi", "fastapi", "serving"),                       ("pydantic", "pydantic", "schemas, validation"),
    ("litellm", "litellm", "multi-provider gateway"),        ("mcp", "mcp", "Model Context Protocol"),
]

rows = []
for module, dist, used_for in PACKAGES:
    t0 = time.perf_counter()
    try:
        importlib.import_module(module)
        status = "ok"
    except Exception as e:                   # ImportError, or a broken install (DLL errors etc.)
        status = f"MISSING ({type(e).__name__})"
    seconds = time.perf_counter() - t0
    try:
        version = md.version(dist)
    except md.PackageNotFoundError:
        version = "-"
    rows.append((module.split(".")[0], dist, version, status, seconds, used_for))

print(f"{'package':22} {'version':12} {'status':10} {'import s':>8}  used for")
for name, dist, version, status, seconds, used_for in rows:
    print(f"{name:22} {version:12} {status:10} {seconds:8.2f}  {used_for}")

missing = [dist for _, dist, _, status, _, _ in rows if status != "ok"]
print(f"\n{len(rows) - len(missing)}/{len(rows)} packages import cleanly.")
if missing:
    print("Fix with:  pip install " + " ".join(missing))

Where does the wait go? This is the first chart of the set, and it shows why the first cell of a Hugging Face notebook takes a few seconds while the rest are instant.

In [ ]:
import matplotlib.pyplot as plt

by_time = sorted(rows, key=lambda r: r[4])
names, secs = [r[0] for r in by_time], [r[4] for r in by_time]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(names, secs, color="#2a78d6", height=0.6)
ax.tick_params(axis="y", labelsize=8)
for name, s in zip(names[-3:], secs[-3:]):                 # label only the three slowest
    ax.text(s, name, f" {s:.1f}s", va="center", fontsize=9, color="#52514e")
ax.set_xlabel("seconds to import (first time in this kernel, incremental)")
ax.set_title("Import time per package: the deep-learning stack dominates")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## Which LLM do the notebooks call?

Every notebook that needs a model gets it from [`llm_setup.py`](../llm_setup.py) at the root of this folder. You get back a real `openai.OpenAI` client, so the code you practise is the code you would ship. What sits behind it:

| Provider | When it is used | Cost | Answers |
|---|---|---|---|
| **offline** (default) | no key and no Ollama | free | a deterministic stand-in inside the process: canned, not intelligent, but every cell runs |
| **openai** | `OPENAI_API_KEY` is set | cents per notebook with `gpt-4.1-mini` | real |
| **ollama** | an Ollama server is running locally | free, private | real, from a small local model (`qwen2.5:3b`) |

```
LLM_PROVIDER set? --yes--> use it (falls back to offline if its key / server is missing)
      | no
OPENAI_API_KEY set? --yes--> openai  (gpt-4.1-mini, text-embedding-3-small)
      | no
Ollama answering on localhost:11434? --yes--> ollama  (qwen2.5:3b, nomic-embed-text)
      | no
offline stand-in
```

**OpenAI.** Create `jupyter_notebooks/.env` (next to `llm_setup.py`). If there is a `.env.example`, copy it. Comments can go on their own lines or after a value (`KEY=value  # note`); quote a value that itself contains ` #`:

```
# jupyter_notebooks/.env  -  gitignored, never commit it
OPENAI_API_KEY=sk-...
# optional overrides:
# LLM_PROVIDER=openai
# OPENAI_MODEL=gpt-4.1-mini
```

`.env` is listed in the repo's `.gitignore`. Never paste a key into a notebook cell: outputs and cells end up in git, screenshots and chat logs.

**Ollama (local, free).** Install it from [ollama.com](https://ollama.com). On Windows it runs as a background app that serves `http://localhost:11434`. Then pull the two models once (about 2 GB and 0.3 GB):

```bat
ollama pull qwen2.5:3b
ollama pull nomic-embed-text
ollama list
```

**Force a provider** with `LLM_PROVIDER=openai|ollama|offline`, either in `.env` or in the notebook *before* the import: `import os; os.environ["LLM_PROVIDER"] = "ollama"`. The choice is made once, at import time, so after changing it you **restart the kernel**. The test runner (`run_all_notebooks.py`) always forces `offline`, so tests never spend money.

### The standard first cell

Every notebook that calls a model starts with exactly this cell. The loop walks up from the notebook's folder until it finds `llm_setup.py`, so it works from any section. The import runs `describe()` for you, which prints the provider and the models. Call `describe()` again any time you want to know where requests are going.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
from llm_setup import describe, ollama_running   # describe() already ran during the import above

env_file = next((p / ".env" for p in [Path.cwd(), *Path.cwd().parents] if (p / "llm_setup.py").exists()), None)
print(f"PROVIDER     : {PROVIDER}  (chat: {MODEL}, embeddings: {EMBED_MODEL})")
print(f".env file    : {'present' if env_file and env_file.exists() else 'not found'}")
print(f"OpenAI key   : {'set' if os.getenv('OPENAI_API_KEY') else 'not set'}   (never print the key itself)")
print(f"Ollama up    : {ollama_running()}")

### One live call, and what you pay for (`fo21`)

A chat call sends a list of **messages** (system, user, assistant, tool) and some parameters (`model`, `max_tokens`, `temperature`, `tools`, `response_format`). It returns **choices** and a **usage** block. You are billed per input token and per output token, and output tokens cost more. At OpenAI's list price for `gpt-4.1-mini` that is $0.40 per million input tokens and $1.60 per million output tokens (check the pricing page, because prices change).

Offline, the reply below is a canned line from the stand-in. With a real model you get a one-sentence definition of RAG. The code is identical either way.

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "In one sentence: what is retrieval-augmented generation?"},
    ],
    max_tokens=60,
)
usage = resp.usage
print("reply  :", resp.choices[0].message.content)
print("model  :", resp.model, "| finish_reason:", resp.choices[0].finish_reason)
print(f"tokens : {usage.prompt_tokens} in + {usage.completion_tokens} out = {usage.total_tokens}")

PRICE_IN, PRICE_OUT = 0.40 / 1e6, 1.60 / 1e6          # USD per token, gpt-4.1-mini list price
cost = usage.prompt_tokens * PRICE_IN + usage.completion_tokens * PRICE_OUT
print(f"cost   : ${cost:.6f} at gpt-4.1-mini prices  ->  ${cost * 1e6:,.0f} per million such calls")

assert isinstance(resp.choices[0].message.content, str) and resp.choices[0].message.content
assert usage.total_tokens == usage.prompt_tokens + usage.completion_tokens

Embeddings use the same client. The offline stand-in hashes words and knows a few synonyms (refund ≈ reimbursement), so a lexical-plus-synonym similarity still behaves sensibly. A real embedding model captures much more meaning. The claim checked below holds for all three providers:

In [ ]:
import numpy as np
from llm_setup import embed

texts = ["I want a refund for my order", "Can I get a reimbursement for my order?", "The weather is sunny today"]
E = np.array(embed(texts))
E = E / np.linalg.norm(E, axis=1, keepdims=True)       # L2-normalise so dot product = cosine similarity
sims = E @ E.T
print(f"embedding size: {E.shape[1]} dims (model: {EMBED_MODEL})")
print(f"refund vs reimbursement : {sims[0, 1]:.2f}")
print(f"refund vs weather       : {sims[0, 2]:.2f}")
assert sims[0, 1] > sims[0, 2]

### The offline stand-in's knobs

When `PROVIDER == "offline"`, `llm_setup.OFFLINE` exposes a few knobs that later notebooks use to simulate production behaviour:

| Knob | Effect | Used in |
|---|---|---|
| `OFFLINE.latency = 0.3` | every request takes 0.3 s (async requests overlap) | async / concurrency |
| `OFFLINE.token_delay = 0.02` | delay between streamed chunks | streaming, time to first token |
| `OFFLINE.fail_next(2, 429)` | the next 2 requests fail with HTTP 429 | retries, backoff, circuit breakers |
| `OFFLINE.request_count` | how many requests were served | caching (did we call the model?) |

Always guard knob use with `if OFFLINE:`. With a real provider it is `None`. The cell below also shows that the OpenAI SDK **retries 429s and 5xx errors twice by default** (`max_retries=2`, exponential backoff), which is easy to miss.

In [ ]:
import openai
from llm_setup import OFFLINE, chat, make_client

if OFFLINE:
    OFFLINE.latency = 0.3
    t0 = time.perf_counter(); chat("ping")
    print(f"with latency=0.3   : one call took {time.perf_counter() - t0:.2f}s")
    OFFLINE.latency = 0.0

    no_retry = make_client(max_retries=0)
    OFFLINE.fail_next(1, 429)
    try:
        no_retry.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "hi"}])
    except openai.RateLimitError as e:
        print(f"max_retries=0      : {type(e).__name__} (HTTP {e.status_code}) reaches your code")

    OFFLINE.fail_next(1, 429)
    t0 = time.perf_counter()
    client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "hi"}])
    print(f"default client     : the 429 was retried silently; call took {time.perf_counter() - t0:.2f}s")
    print(f"requests served    : {OFFLINE.request_count}")
else:
    print(f"PROVIDER={PROVIDER}: the knobs exist only for the offline stand-in.")

## Roadmap: the sections in order

| # | Section | What you will be able to do | Start with |
|---|---|---|---|
| 00 | Start here | set up, switch providers, answer the 12 most frequent quick topics | [quick interview topics](01_quick_interview_topics.ipynb) |
| 01 | Python basics | handle the language questions: types, collections, functions, OOP, errors, typing & Pydantic, async, gotchas | [variables & types](../01_python_basics/01_variables_types_numbers.ipynb) |
| 02 | Python problems | solve warm-ups and AI-flavoured tasks such as parsing LLM JSON, chunking and backoff | [warm-ups](../02_python_problems/01_warmup_problems.ipynb) |
| 03 | DSA | recognise the 14 patterns behind coding rounds, and time them | [Big-O & timing](../03_dsa/01_big_o_and_timing.ipynb) |
| 04 | NumPy, pandas, PyTorch | use the daily tools: arrays & broadcasting, DataFrames & logs, tensors & training loops, charts | [NumPy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb) |
| 05 | Machine learning | split, fit, measure, regularise and pipeline classical models | [ML workflow](../05_machine_learning/01_ml_workflow_and_train_test_split.ipynb) |
| 06 | Deep learning | do backprop by hand, then optimisers, regularisation, CNNs, and embeddings to attention | [neuron & backprop](../06_deep_learning/01_neuron_and_backprop_from_scratch.ipynb) |
| 07 | GenAI foundations | explain and build tokenization, embeddings, attention, a tiny GPT, decoding, KV cache, quantization | [tokenization](../07_genai_foundations/01_tokenization_bpe_and_tiktoken.ipynb) |
| 08 | LLM apps | call the API, prompt, get structured outputs, call functions, run local models, manage memory | [OpenAI API basics](../08_llm_apps/01_openai_api_basics.ipynb) |
| 09 | RAG | chunk, search with BM25 and vectors, fuse with RRF, rerank, and build the full pipeline | [chunking](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb) |
| 10 | Hugging Face | use the Hub & pipelines, tokenizers & `generate`, datasets, sentence-transformers, the Trainer, bitsandbytes | [Hub & pipelines](../10_huggingface/01_hub_and_pipelines.ipynb) |
| 11 | Fine-tuning | decide when to fine-tune, then run SFT with loss masking, LoRA, QLoRA, DPO and distillation | [when to fine-tune](../11_finetuning/01_when_to_fine_tune_and_data_prep.ipynb) |
| 12 | Evaluation | measure with EM/F1/ROUGE, LLM judges, RAG and agent evals, and A/B statistics | [eval metrics](../12_evaluation/01_llm_eval_metrics_em_f1_bleu_rouge.ipynb) |
| 13 | LangChain | work with models & prompts, LCEL, structured output, retrievers, the RAG chain, agents | [models & prompts](../13_langchain/01_models_messages_and_prompts.ipynb) |
| 14 | LangGraph | build state graphs, loops, checkpoints, human-in-the-loop, ReAct agents and supervisors | [state, nodes, edges](../14_langgraph/01_state_nodes_and_edges.ipynb) |
| 15 | Agentic AI | build the agent loop from scratch, then tools, planning, memory, multi-agent, guardrails, MCP, System-1 models | [agent loop](../15_agentic_ai/01_agent_loop_from_scratch.ipynb) |
| 16 | Production | handle async, retries, caching, streaming APIs, p95 & cost, LiteLLM routing, PII, tracing | [async concurrency](../16_production/01_async_concurrency_for_llm_calls.ipynb) |
| 17 | Coding rounds | pass the blank-editor rounds: RAG skeleton, agent in 30 minutes, chat API, search service, eval harness | [RAG skeleton](../17_interview_coding_rounds/01_rag_skeleton.ipynb) |

### Three study tracks

**Track A: 2-week crash.** Two notebooks a day, the minimum to hold your own in a GenAI engineer loop.

1. Day 1: [quick interview topics](01_quick_interview_topics.ipynb) · [NumPy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb)
2. Day 2: [tokenization](../07_genai_foundations/01_tokenization_bpe_and_tiktoken.ipynb) · [embeddings & similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb)
3. Day 3: [attention from scratch](../07_genai_foundations/03_attention_from_scratch.ipynb) · [decoding: temperature, top-k, top-p](../07_genai_foundations/05_decoding_temperature_top_k_top_p.ipynb)
4. Day 4: [KV cache](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb) · [quantization](../07_genai_foundations/07_quantization_int8_int4_nf4.ipynb)
5. Day 5: [OpenAI API basics](../08_llm_apps/01_openai_api_basics.ipynb) · [structured outputs with Pydantic](../08_llm_apps/03_structured_outputs_with_pydantic.ipynb)
6. Day 6: [prompt engineering](../08_llm_apps/02_prompt_engineering.ipynb) · [function calling](../08_llm_apps/04_function_calling.ipynb)
7. Day 7: [chunking](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb) · [hybrid search & RRF](../09_rag/04_hybrid_search_and_rrf.ipynb)
8. Day 8: [reranking](../09_rag/05_reranking.ipynb) · [RAG pipeline from scratch](../09_rag/06_rag_pipeline_from_scratch.ipynb)
9. Day 9: [eval metrics](../12_evaluation/01_llm_eval_metrics_em_f1_bleu_rouge.ipynb) · [RAG evaluation](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb)
10. Day 10: [agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb) · [LangGraph ReAct agent](../14_langgraph/05_react_tool_calling_agent.ipynb)
11. Day 11: [when to fine-tune](../11_finetuning/01_when_to_fine_tune_and_data_prep.ipynb) · [LoRA from scratch](../11_finetuning/03_lora_from_scratch.ipynb)
12. Day 12: [retries, backoff, circuit breakers](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb) · [p50/p95 & cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb)
13. Day 13: [RAG skeleton round](../17_interview_coding_rounds/01_rag_skeleton.ipynb) · [agent in 30 minutes](../17_interview_coding_rounds/02_build_an_agent_in_30_minutes.ipynb)
14. Day 14: [caching](../16_production/03_caching_exact_and_semantic.ipynb) · [implement-from-scratch drills](../17_interview_coding_rounds/07_implement_from_scratch_drills.ipynb)

**Track B: GenAI / RAG engineer.**

1. [NumPy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb)
2. [pandas essentials](../04_numpy_pandas_pytorch/02_pandas_essentials.ipynb)
3. [tokenization](../07_genai_foundations/01_tokenization_bpe_and_tiktoken.ipynb)
4. [embeddings & similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb)
5. [OpenAI API basics](../08_llm_apps/01_openai_api_basics.ipynb)
6. [prompt engineering](../08_llm_apps/02_prompt_engineering.ipynb)
7. [structured outputs](../08_llm_apps/03_structured_outputs_with_pydantic.ipynb)
8. [chunking & semantic chunking](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb)
9. [BM25 from scratch](../09_rag/02_bm25_from_scratch.ipynb)
10. [vector search: FAISS & HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb)
11. [hybrid search & RRF](../09_rag/04_hybrid_search_and_rrf.ipynb)
12. [reranking](../09_rag/05_reranking.ipynb)
13. [RAG pipeline from scratch](../09_rag/06_rag_pipeline_from_scratch.ipynb)
14. [advanced RAG: rewriting, HyDE, multi-query](../09_rag/07_advanced_rag_query_rewriting_hyde_multiquery.ipynb)
15. [sentence-transformers & cross-encoders](../10_huggingface/04_sentence_transformers_embeddings_and_cross_encoders.ipynb)
16. [eval metrics](../12_evaluation/01_llm_eval_metrics_em_f1_bleu_rouge.ipynb)
17. [LLM-as-judge and its biases](../12_evaluation/02_llm_as_judge_and_its_biases.ipynb)
18. [RAG evaluation](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb)
19. [LangChain retrievers](../13_langchain/04_loaders_splitters_vectorstores_retrievers.ipynb)
20. [LangChain RAG chain](../13_langchain/05_rag_chain_end_to_end.ipynb)
21. [caching](../16_production/03_caching_exact_and_semantic.ipynb)
22. [FastAPI serving & streaming](../16_production/04_fastapi_serving_and_streaming.ipynb)
23. [p50/p95 & cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb)
24. [RAG skeleton round](../17_interview_coding_rounds/01_rag_skeleton.ipynb)
25. [semantic search service round](../17_interview_coding_rounds/04_semantic_search_service.ipynb)
26. [eval harness round](../17_interview_coding_rounds/05_eval_harness_from_scratch.ipynb)

**Track C: agentic engineer.**

1. [OpenAI API basics](../08_llm_apps/01_openai_api_basics.ipynb)
2. [structured outputs](../08_llm_apps/03_structured_outputs_with_pydantic.ipynb)
3. [function calling](../08_llm_apps/04_function_calling.ipynb)
4. [conversation memory & context windows](../08_llm_apps/06_conversation_memory_and_context_windows.ipynb)
5. [agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb)
6. [tool design & function schemas](../15_agentic_ai/02_tool_design_and_function_schemas.ipynb)
7. [ReAct & plan-and-execute](../15_agentic_ai/03_react_and_plan_and_execute.ipynb)
8. [agent memory](../15_agentic_ai/04_agent_memory_short_and_long_term.ipynb)
9. [LangGraph: state, nodes, edges](../14_langgraph/01_state_nodes_and_edges.ipynb)
10. [conditional edges & loops](../14_langgraph/02_conditional_edges_and_loops.ipynb)
11. [checkpointing & time travel](../14_langgraph/03_checkpointing_memory_and_time_travel.ipynb)
12. [human-in-the-loop interrupts](../14_langgraph/04_human_in_the_loop_interrupts.ipynb)
13. [ReAct tool-calling agent](../14_langgraph/05_react_tool_calling_agent.ipynb)
14. [multi-agent supervisor](../14_langgraph/06_multi_agent_supervisor.ipynb)
15. [multi-agent patterns](../15_agentic_ai/05_multi_agent_patterns.ipynb)
16. [guardrails, HITL & failure modes](../15_agentic_ai/06_guardrails_hitl_and_failure_modes.ipynb)
17. [MCP](../15_agentic_ai/07_mcp_model_context_protocol.ipynb)
18. [System-1 decision models: Jev & Laya](../15_agentic_ai/08_system1_decision_models_jev_and_laya.ipynb)
19. [LangChain agents & memory](../13_langchain/06_agents_and_memory.ipynb)
20. [agent evaluation](../12_evaluation/04_agent_evaluation.ipynb)
21. [async concurrency](../16_production/01_async_concurrency_for_llm_calls.ipynb)
22. [retries & circuit breakers](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb)
23. [LiteLLM gateway & routing](../16_production/06_litellm_gateway_and_model_routing.ipynb)
24. [guardrails: PII & prompt injection](../16_production/07_guardrails_pii_and_prompt_injection.ipynb)
25. [observability & tracing](../16_production/08_observability_logging_and_tracing.ipynb)
26. [agent in 30 minutes round](../17_interview_coding_rounds/02_build_an_agent_in_30_minutes.ipynb)

## Try it yourself

**1.** The check above skipped `accelerate`, `httpx` and `nbclient`. Print their versions without importing them. (Hint: `importlib.metadata`.)

In [ ]:
# Solution — try it yourself first, then run this
for dist in ["accelerate", "httpx", "nbclient"]:
    try:
        print(f"{dist:12} {md.version(dist)}")
    except md.PackageNotFoundError:
        print(f"{dist:12} not installed")

**2.** Write `ask(prompt)` that returns `(reply_text, cost_usd)` using the `usage` block and the gpt-4.1-mini prices above. Then check that a longer `max_tokens` can only raise the worst-case cost.

In [ ]:
# Solution — try it yourself first, then run this
def ask(prompt: str, max_tokens: int = 100) -> tuple[str, float]:
    r = client.chat.completions.create(model=MODEL, max_tokens=max_tokens,
                                       messages=[{"role": "user", "content": prompt}])
    u = r.usage
    return r.choices[0].message.content, u.prompt_tokens * PRICE_IN + u.completion_tokens * PRICE_OUT

text, usd = ask("Name three uses of embeddings.")
print(f"reply: {text[:80]!r}")
print(f"cost : ${usd:.6f}")
worst_case = lambda prompt_tokens, max_tokens: prompt_tokens * PRICE_IN + max_tokens * PRICE_OUT
assert worst_case(50, 500) > worst_case(50, 100)

**3.** Predict first: with `OFFLINE.latency = 0.2`, how long do three calls in a plain `for` loop take? Time them. (The async notebook in section 16 makes them overlap.)

In [ ]:
# Solution — try it yourself first, then run this
if OFFLINE:
    OFFLINE.latency = 0.2
    t0 = time.perf_counter()
    for q in ["a", "b", "c"]:
        chat(q)
    took = time.perf_counter() - t0
    OFFLINE.latency = 0.0
    print(f"3 sequential calls: {took:.2f}s (about 3 x 0.2s: each waits for the previous one)")
    assert took >= 0.6
else:
    print("Only meaningful with the offline stand-in.")

**4.** Prove that your key cannot be committed: walk up from this folder, find the repo's `.gitignore`, and check that it ignores `.env`.

In [ ]:
# Solution — try it yourself first, then run this
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), None)
gitignore = repo_root / ".gitignore" if repo_root else None
if gitignore and gitignore.exists():
    ignored = {line.strip() for line in gitignore.read_text(encoding="utf-8").splitlines()}
    print(f"repo root  : {repo_root.name}")
    print(f".env ignored: {'.env' in ignored}")
else:
    print("No git repo found above this folder (copied elsewhere?) - add `.env` to your .gitignore yourself.")

## Say it in an interview

**"How do you test LLM code in CI without spending money?"** (`pd29`). Fake the model at the transport layer so the real SDK code path runs. That can be an httpx mock transport like the stand-in here, `mock_response=` in LiteLLM, or recorded fixtures. Assert on structure (valid JSON, the right tool called, citations present), not on exact wording. Keep a small, separately budgeted job that runs the real model on a golden set.

**"How would you migrate from one provider to another?"** (`pd16`). Most providers and local servers (Ollama, vLLM) speak the OpenAI Chat Completions API. The code change is `base_url` plus `model`, and here it is none at all. The real work is behavioural: re-run your evals, because JSON reliability, tool-calling quality, context length and refusal style all differ. Keep prompts and model names in config, and consider a gateway such as LiteLLM so fallbacks are configuration.

**"Where do the keys live?"** (`pd25`). Locally they go in environment variables or a gitignored `.env`. In production use a secret manager with per-environment keys, least privilege and rotation. Never put them in code, notebook outputs, logs or prompts. Scan commits for leaked secrets.

**"Would you use Ollama in production?"** (`tl11`). It is great for development, privacy-sensitive prototypes, offline demos and low-traffic internal tools. For high-concurrency serving, pick an engine built for throughput (vLLM, TGI, SGLang) with continuous batching and paged KV cache.

**"What are you charged for?"** (`fo21`). Input tokens plus output tokens, with output priced higher (4x for gpt-4.1-mini). Cached input is cheaper, and batch APIs are cheaper still. The `usage` block on every response is your meter.

**Traps.** `temperature=0` is not a determinism guarantee. The OpenAI SDK silently retries 429s and 5xx errors twice unless you set `max_retries`. Changing `LLM_PROVIDER` does nothing until you restart the kernel. The stand-in's answers are canned: never read quality into them.

## Next

- [01 · Quick interview topics](01_quick_interview_topics.ipynb): the 12 topics that come up constantly, each with a demo.
- [OpenAI API basics](../08_llm_apps/01_openai_api_basics.ipynb) and [local LLMs with Ollama](../08_llm_apps/05_local_llms_with_ollama.ipynb) go deeper on the provider switch.
- [LiteLLM gateway & model routing](../16_production/06_litellm_gateway_and_model_routing.ipynb): one API for many providers, with fallbacks.
- [NumPy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb) starts the hands-on track.
- Theory: [interview bank](../../ai_interview_prep/index.html) · [genai_flow course](../../genai_flow/index.html) · [INTERVIEW_MAP.md](../INTERVIEW_MAP.md)